# 04 | Longitudinal preparation
**NHS A&E Intelligent Decision Support System**

**Objective:** align each provider's reporting calendar and define provisional next-month Type 1 forecasting eligibility without future observations.

**Execution:** explore reads saved evidence; rebuild runs the stages in an isolated local workspace; snapshot generates a versioned research run. See the project README for the mode settings.

## 1. Methodology and parameters

Each provider enters the calendar in its first observed month. Subsequent months without a record are marked absent, even after its last observation. No counts or provider attributes are filled. These calendar rows are analytical placeholders, not NHS submissions.

The eligibility rule is provisional: the provider must have an observation at the origin, 12 consecutive observed months through the origin, and at least one positive Type 1 value within that window. This is a proxy for recent activity, not verified service classification. A 12-month window supports evaluating a seasonal baseline. The rule is fixed for this analysis and can be evaluated for sensitivity using training data only.

The origin means the latest available reporting month. The actual publication date and data availability must be considered before building forecasting features.

In [ ]:
from pathlib import Path
import sys

MODE = "explore"  # explore, rebuild, snapshot
VIEW_REBUILD = False  # In explore mode, view completed local rebuild results.
RESET_REBUILD = False  # Notebook 01 only: explicitly replace the local rebuild.
EXPLORE_RUN_IDS = {}  # Optional saved run IDs by milestone, e.g. {"M03": "..."}.

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/utils/execution_modes.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.utils.execution_modes import NotebookSession

session = NotebookSession(project_root, MODE, 4, reset_rebuild=RESET_REBUILD,
                          view_rebuild=VIEW_REBUILD, run_ids=EXPLORE_RUN_IDS)
repo_root = session.repo
print("Execution mode:", MODE)
if MODE == "explore":
    print("Saved evidence:", session.locations())
else:
    from pathlib import Path
    import sys
    from IPython.display import display, Markdown, Image
    import pandas as pd

    from src.data.longitudinal_preparation import run_longitudinal, TARGET, STATUSES

    M04_RUN_ID = session.prior_run('M04')
    HISTORY_WINDOW = 12


## 2. Execute longitudinal preparation

The pipeline checks that the selected M04 run is complete and that the source output fingerprints match its recorded values. It checks row conservation, calendar uniqueness and preservation of reported counts. It then runs regression tests and exports a timestamped evidence snapshot.

In [ ]:
if MODE == "explore":
    session.show('M05', 'reports/m05_summary.json')
else:
    result = run_longitudinal(repo_root, M04_RUN_ID, HISTORY_WINDOW, run_tests=True, notebook=True)
    s = result["summary"]
    panel, profile, monthly = (result[k] for k in ("panel", "profile", "monthly"))
    evidence = Path(result["evidence_path"])
    if result["evidence_status"] != "COMPLETE":
        raise RuntimeError("Evidence run requires review. Do not use partial outputs.")
    display(Markdown(f"""### Results at a glance

| Measure | Result |
|---|---:|
| Reporting months | {s['reporting_months']:,} |
| Provider codes observed at least once | {s['provider_codes']:,} |
| Original provider-month records conserved | {s['observed_rows']:,} |
| Calendar rows, including absent records | {s['calendar_rows']:,} |
| Absent records | {s['absent_rows']:,} |
| Reported zero Type 1 observations | {s['reported_zero_rows']:,} |
| Provisional candidates in {s['latest_month']} | {s['latest_candidates']:,} / {s['latest_observed_providers']:,} reported providers |

**Pipeline outcome:** `{s['status']}`. This does not establish clinical equivalence of provider codes or model readiness."""))


## 3. Reporting coverage and missing months

**Research question:** how stable is the reporting population after providers first appear?

**Method:** compare rows in the provider calendar with actual reported observations. The gap between the two series represents absent records, not zero attendances or confirmed service closures.

In [ ]:
if MODE == "explore":
    session.show('M05', 'figures/01_reporting_coverage.png')
else:
    display(Image(filename=str(evidence / "figures/01_reporting_coverage.png"), width=900))
    display(Markdown(f"""**Finding:** {s['observed_rows']:,} reported records were retained. The calendar additionally identifies {s['absent_rows']:,} absent records: {s['internal_gap_rows']:,} within observed spans and {s['absent_after_last_observation_rows']:,} after final observations.

**Interpretation:** reporting absence may reflect organisational changes, a service change or a reporting gap. The dataset alone does not establish the cause.

**Decision:** retain null counts for absent records. Never substitute zero or use the next available row as if it were the next calendar month."""))


## 4. Provider continuity and identity

**Research question:** which codes have reporting gaps or variations in recorded names and parent groups?

**Method:** count internal gaps retrospectively within each provider's first and last observed month. Review recorded name and parent-group variants without assuming they imply mergers. This descriptive audit uses full history; it must not be used to select earlier model cohorts.

In [ ]:
if MODE == "explore":
    session.show('M05', 'figures/03_provider_gap_review.png', optional=True)
else:
    display(Image(filename=str(evidence / "figures/03_provider_gap_review.png"), width=860))
    review = profile.loc[(profile.missing_within_observed_span > 0) | (profile.distinct_recorded_names > 1) | (profile.distinct_recorded_parent_groups > 1)]
    display(review.sort_values(["missing_within_observed_span", "org_code"], ascending=[False, True]).head(12)[["org_code", "first_observed_month", "last_observed_month", "observed_months", "missing_within_observed_span", "distinct_recorded_names", "distinct_recorded_parent_groups"]])
    display(Markdown(f"""**Finding:** {s['providers_with_internal_gaps']:,} codes have internal gaps; {s['providers_with_name_variants']:,} have recorded name variants and {s['providers_with_parent_variants']:,} have parent-group variants.

**Decision:** investigate relevant identities using historical NHS organisation and service information. Do not merge or discard records based solely on name variation. Retain the M03 change advisories for source-based review."""))


## 5. Time-aware forecasting eligibility

**Research question:** which provider-month origins have sufficient recent observations to support a next-month forecast?

**Method:** rolling calendar windows use observed records through the origin, including the origin itself. Candidate status does not use the next-month outcome. A provider may be a candidate even when its origin value is zero if the preceding window contains positive activity. A provider with a missing month is not eligible until the required consecutive history is re-established.

In [ ]:
if MODE == "explore":
    session.show('M05', 'figures/02_time_aware_candidates.png')
else:
    display(Image(filename=str(evidence / "figures/02_time_aware_candidates.png"), width=900))
    latest = monthly.iloc[-1]
    status_table = pd.DataFrame({"Status": STATUSES, "Provider codes": [int(latest[k]) for k in STATUSES]})
    display(status_table)
    display(Markdown(f"""**Finding:** at the latest reporting month ({s['latest_month']}), {s['latest_candidates']:,} of {s['latest_observed_providers']:,} reported providers meet the provisional candidate rule. The {s['candidate_origin_rows']:,} candidate origins across the history are not the number of usable training pairs.

**Decision:** preserve all observations and evaluate seasonal-naive and regression models on comparable, time-aware cohorts. At model evaluation, require a real next-month target, never a filled value. Verify source publication timing before assuming a reporting month was available at an origin."""))


## 6. Decision register and evidence

The register distinguishes implemented data transformations from assumptions and outstanding source reviews. M06 will establish the final analytical data contract before M07 creates features.

In [ ]:
if MODE == "explore":
    session.show('M05', 'reports/m05_decisions.csv')
    print('Saved evidence:', session.locations())
else:
    display(result["decisions"])
    print("Processed outputs:")
    for path in s["outputs"]:
        print(path)
    print("Evidence:", evidence.relative_to(repo_root))
    print("Manifest status:", result["evidence_status"])
    session.finish()
    print('Stage completed.')
